# Training Gold publishing

Reads the training Silver table and writes Gold current/route outputs with the same column shapes as `gold_vehicle_positions_current` and `gold_route_direction_5min`.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

TRAINING_SILVER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.training_silver_hsl_vehicle_position"
TRAINING_GOLD_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.training_gold_hsl_vehicle_position"
TRAINING_GOLD_CURRENT_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.training_gold_hsl_vehicle_position_current"
TRAINING_GOLD_ROUTE_5MIN_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.training_gold_hsl_vehicle_position_route_5min"

TRAINING_GOLD_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/gold/hsl_vehicle_position"
TRAINING_GOLD_CURRENT_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/gold/hsl_vehicle_position_current"
TRAINING_GOLD_ROUTE_5MIN_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/gold/hsl_vehicle_position_route_5min"

CHECKPOINT_GOLD_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/checkpoints/gold_hsl_vehicle_position"
CHECKPOINT_GOLD_CURRENT_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/checkpoints/gold_hsl_vehicle_position_current"
CHECKPOINT_GOLD_ROUTE_5MIN_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/checkpoints/gold_hsl_vehicle_position_route_5min"

TRIGGER_INTERVAL = "10 seconds"
WATERMARK_DELAY = "15 minutes"

RESET_TABLES = False
RESET_CHECKPOINTS = False

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name in {
        "training_gold_hsl_vehicle_position",
        "training_gold_hsl_vehicle_position_current",
        "training_gold_hsl_vehicle_position_route_5min",
    }:
        q.stop()

if RESET_TABLES:
    for table_name, path in [
        (TRAINING_GOLD_TABLE, TRAINING_GOLD_PATH),
        (TRAINING_GOLD_CURRENT_TABLE, TRAINING_GOLD_CURRENT_PATH),
        (TRAINING_GOLD_ROUTE_5MIN_TABLE, TRAINING_GOLD_ROUTE_5MIN_PATH),
    ]:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
        dbutils.fs.rm(path, True)

if RESET_CHECKPOINTS:
    for path in [CHECKPOINT_GOLD_PATH, CHECKPOINT_GOLD_CURRENT_PATH, CHECKPOINT_GOLD_ROUTE_5MIN_PATH]:
        dbutils.fs.rm(path, True)


In [ ]:
def classify_delay(col_expr):
    return (
        F.when(col_expr.isNull(), F.lit("unknown"))
        .when(col_expr <= F.lit(-60), F.lit("early"))
        .when(col_expr <= F.lit(120), F.lit("on_time"))
        .when(col_expr <= F.lit(300), F.lit("delayed"))
        .otherwise(F.lit("severely_delayed"))
    )


def classify_occupancy(col_expr):
    return (
        F.when(col_expr.isNull(), F.lit("unknown"))
        .when(col_expr <= F.lit(20), F.lit("empty_or_low"))
        .when(col_expr <= F.lit(50), F.lit("moderate"))
        .when(col_expr <= F.lit(80), F.lit("busy"))
        .otherwise(F.lit("crowded"))
    )


silver_stream_df = spark.readStream.table(TRAINING_SILVER_TABLE)

gold_business_df = (
    silver_stream_df
    .withColumn("canonical_route_id", F.coalesce(F.col("route_id"), F.col("topic_route_id"), F.col("payload_route_id"), F.col("line_id")))
    .withColumn("canonical_direction_id", F.coalesce(F.col("direction_id"), F.col("topic_direction_id"), F.col("dir")))
    .withColumn("gold_publish_ts", F.current_timestamp())
    .withColumn("gold_service_ts", F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("silver_ingest_ts"), F.current_timestamp()))
    .withColumn("service_date", F.coalesce(F.col("operating_day"), F.to_date(F.col("event_ts")), F.col("silver_event_date")))
    .withColumn("service_hour", F.hour(F.col("gold_service_ts")))
    .withColumn("route_direction_key", F.concat_ws("|", F.coalesce(F.col("canonical_route_id"), F.lit("")), F.coalesce(F.col("canonical_direction_id"), F.lit(""))))
    .withColumn("route_vehicle_key", F.concat_ws("|", F.coalesce(F.col("canonical_route_id"), F.lit("")), F.coalesce(F.col("vehicle_id"), F.lit(""))))
    .withColumn("delay_status", classify_delay(F.col("delay_sec")))
    .withColumn("occupancy_status", classify_occupancy(F.col("occupancy")))
    .withColumn(
        "location_quality",
        F.when(F.col("latitude").isNull() | F.col("longitude").isNull(), F.lit("missing"))
        .when((F.col("latitude") < F.lit(59.0)) | (F.col("latitude") > F.lit(61.5)) | (F.col("longitude") < F.lit(23.0)) | (F.col("longitude") > F.lit(26.5)), F.lit("out_of_bounds"))
        .otherwise(F.lit("ok"))
    )
    .withColumn("is_delayed", F.when(F.col("delay_sec").isNull(), F.lit(False)).otherwise(F.col("delay_sec") > F.lit(120)))
    .withColumn("is_severely_delayed", F.when(F.col("delay_sec").isNull(), F.lit(False)).otherwise(F.col("delay_sec") > F.lit(300)))
    .withColumn("has_coordinates", F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .withColumn("gold_event_date", F.to_date(F.col("gold_service_ts")))
    .withColumn("event_to_gold_delay_sec", (F.col("gold_publish_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
    .withColumn("silver_to_gold_delay_sec", (F.col("gold_publish_ts").cast("long") - F.col("silver_ingest_ts").cast("long")).cast("long"))
    .withColumn("gold_record_type", F.lit("vehicle_position"))
)


In [ ]:
def register_table(path: str, table_name: str):
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{path}'")


def precreate_sink(path: str, table_name: str, schema, partition_columns=None):
    if not DeltaTable.isDeltaTable(spark, path):
        writer = (
            spark.createDataFrame([], schema)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        if partition_columns:
            writer = writer.partitionBy(*partition_columns)
        writer.save(path)
    register_table(path, table_name)


precreate_sink(TRAINING_GOLD_PATH, TRAINING_GOLD_TABLE, gold_business_df.schema, partition_columns=["gold_event_date"])
precreate_sink(TRAINING_GOLD_CURRENT_PATH, TRAINING_GOLD_CURRENT_TABLE, gold_business_df.schema, partition_columns=["gold_event_date"])

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TRAINING_GOLD_ROUTE_5MIN_TABLE} (
    window_start_ts TIMESTAMP,
    window_end_ts TIMESTAMP,
    gold_window_date DATE,
    service_date DATE,
    route_id STRING,
    direction_id STRING,
    line_id STRING,
    transport_mode STRING,
    position_rows BIGINT,
    distinct_vehicle_count BIGINT,
    avg_speed DOUBLE,
    avg_delay_sec DOUBLE,
    max_delay_sec INT,
    avg_occupancy DOUBLE,
    delayed_rows BIGINT,
    delayed_ratio DOUBLE,
    missing_coordinate_rows BIGINT,
    snapshot_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (gold_window_date)
LOCATION "{TRAINING_GOLD_ROUTE_5MIN_PATH}"
''')


def upsert_gold_current(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    latest_batch_df = (
        batch_df
        .filter(F.col("vehicle_id").isNotNull() & (F.trim(F.col("vehicle_id")) != ""))
        .withColumn(
            "latest_rank",
            F.row_number().over(
                Window.partitionBy("vehicle_id").orderBy(
                    F.col("gold_service_ts").desc_nulls_last(),
                    F.col("gold_publish_ts").desc_nulls_last(),
                    F.col("silver_ingest_ts").desc_nulls_last(),
                )
            )
        )
        .filter(F.col("latest_rank") == 1)
        .drop("latest_rank")
    )

    (
        DeltaTable.forPath(spark, TRAINING_GOLD_CURRENT_PATH)
        .alias("t")
        .merge(latest_batch_df.alias("s"), "t.vehicle_id = s.vehicle_id")
        .whenMatchedUpdateAll(
            condition='''
            coalesce(s.gold_service_ts, s.event_ts, s.eventhub_enqueued_ts, s.gold_publish_ts) >=
            coalesce(t.gold_service_ts, t.event_ts, t.eventhub_enqueued_ts, t.gold_publish_ts)
            '''
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


gold_route_agg_df = (
    gold_business_df
    .withWatermark("gold_service_ts", WATERMARK_DELAY)
    .groupBy(
        F.window("gold_service_ts", "5 minutes"),
        "service_date",
        "canonical_route_id",
        "canonical_direction_id",
        "line_id",
        "transport_mode",
    )
    .agg(
        F.count("*").cast("bigint").alias("position_rows"),
        F.approx_count_distinct("vehicle_id").cast("bigint").alias("distinct_vehicle_count"),
        F.avg("speed").alias("avg_speed"),
        F.avg("delay_sec").alias("avg_delay_sec"),
        F.max("delay_sec").cast("int").alias("max_delay_sec"),
        F.avg("occupancy").alias("avg_occupancy"),
        F.sum(F.when(F.col("is_delayed"), 1).otherwise(0)).cast("bigint").alias("delayed_rows"),
        F.sum(F.when(~F.col("has_coordinates"), 1).otherwise(0)).cast("bigint").alias("missing_coordinate_rows"),
    )
    .select(
        F.col("window.start").alias("window_start_ts"),
        F.col("window.end").alias("window_end_ts"),
        F.to_date(F.col("window.start")).alias("gold_window_date"),
        F.col("service_date"),
        F.col("canonical_route_id").alias("route_id"),
        F.col("canonical_direction_id").alias("direction_id"),
        F.col("line_id"),
        F.col("transport_mode"),
        F.col("position_rows"),
        F.col("distinct_vehicle_count"),
        F.col("avg_speed"),
        F.col("avg_delay_sec"),
        F.col("max_delay_sec"),
        F.col("avg_occupancy"),
        F.col("delayed_rows"),
        (F.col("delayed_rows") / F.col("position_rows").cast("double")).alias("delayed_ratio"),
        F.col("missing_coordinate_rows"),
        F.current_timestamp().alias("snapshot_ts"),
    )
)

gold_stream_query = (
    gold_business_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_PATH)
    .queryName("training_gold_hsl_vehicle_position")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(TRAINING_GOLD_TABLE)
)

gold_current_query = (
    gold_business_df.writeStream
    .foreachBatch(upsert_gold_current)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_CURRENT_PATH)
    .queryName("training_gold_hsl_vehicle_position_current")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

gold_route_query = (
    gold_route_agg_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_ROUTE_5MIN_PATH)
    .queryName("training_gold_hsl_vehicle_position_route_5min")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(TRAINING_GOLD_ROUTE_5MIN_TABLE)
)

print("Training Gold streams started.")
for q in [gold_stream_query, gold_current_query, gold_route_query]:
    print("  Query name:", q.name)
    print("  Query ID  :", q.id)
